In [1]:
# -*- coding: utf-8 -*-
"""
Created on Tue Dec 19 11:46:51 2023

@author: Thoma
"""

import pandas
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time

# Chemin vers le fichier chromedriver (pour Chrome)
driver_path = 'C:/Users/Thoma/AppData/Local/Programs/Spyder/Python/chromedriver'

# Options du navigateur Chrome
chrome_options = webdriver.ChromeOptions()

# chrome_options.add_argument('headless')

# Créer une instance du navigateur
driver = webdriver.Chrome(options=chrome_options)

# Ouvrir la page webhttps://fr.kompass.com/a/services-aux-entreprises/80/v/clermont-ferrand/fr_83_63_63113/
#driver.get("https://www.helloasso.com/e/recherche?tab=associations&bbox=-7.316687829134622%2C42.10613515649942%2C10.657959706435463%2C51.21356451029922&category_tags=sport&category_tags=loisirs")
#driver.get("https://www.helloasso.com/e/recherche?tab=associations&bbox=-6.4760793971694%2C42.1061351564999%2C9.817351274467%2C51.21356451029985&category_tags=sport")
driver.get("https://www.helloasso.com/e/recherche/associations?page=1&category_tags=sport&place_city=Toulouse&place_department=Haute-Garonne")

# Mettre en plein écran la fenêtre du navigateur
driver.maximize_window()
driver.implicitly_wait(12)



def is_element_on_page(driver, xpath):
    try:
        # Recherchez l'élément par XPath
        element = driver.find_element(By.XPATH, xpath)
        
        # Si l'élément est trouvé, renvoie True
        return element is not None
    except:
        # Si une exception est levée (élément non trouvé), renvoie False
        return False






k=1

data_list = []
i=0
t=0

while t < 0 : 
    passer = driver.find_element(By.XPATH, '//*[@id="results"]/div[1]/button[4]')
    driver.implicitly_wait(10)
    time.sleep(3)
    driver.execute_script("arguments[0].click();", passer)
    t+=1
time.sleep(1)
    
while  i<50:
    print(i)
    k=1
    while (k<31) :

        try :
            # Attendez que la page soit complètement chargée (vous pouvez ajuster le délai selon les besoins)
            print(k)
            # Simuler le clic sur l'image
            lien_asso = driver.find_element(By.XPATH, '//*[@id="results"]/ul/li[{}]/a/div/div[1]/p[1]'.format(k))

            nom_association = lien_asso.text.strip()
            nom_association = nom_association.replace(' ', '-')
            nom_association = nom_association.lower()
            nom_association = nom_association.replace('é', 'e')
            nom_association = nom_association.replace('è', 'e')
            nom_association = nom_association.replace('à', 'a')
            nom_association = nom_association.replace("'", '-')
            print(nom_association)
            
            # Ouvrir le lien dans une nouvelle fenêtre
            driver.execute_script(f"window.open('https://www.helloasso.com/associations/{nom_association}', '_blank');")
            
            new_window_handle = driver.window_handles[-1]
            driver.switch_to.window(new_window_handle)
            # Attendre un peu avant de poursuivre
            time.sleep(1)
            if  (is_element_on_page(driver, '//*[@id="carousel-prod_organizations"]/div[1]/div[1]/div/h2') ) :
                # Fermez le navigat/eur
                driver.back()
                # Fermer la nouvelle fenêtre
                driver.close()
                
                # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
                driver.switch_to.window(driver.window_handles[0])
                
                # Basculer vers la fenêtre principale
                k+=1

            else :
                # Simuler le clic sur le bouton "Afficher l'adresse"
                wait = WebDriverWait(driver, 2)
                afficher_adresse_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="contact"]/div[2]/div[2]/div[1]/div[2]/div/div/button')))
                driver.execute_script("arguments[0].click();", afficher_adresse_button)
                # Attendez que la page soit complètement chargée après le deuxième clic
                # Maintenant, vous pouvez extraire les informations de la page, y compris les adresses e-mail
                # Simuler le clic sur un lien avec "email" dans l'attribut href
                email_link = driver.find_element(By.XPATH, '//*[@id="contact"]/div[2]/div[2]/div/div[2]/div/div/p')
                email_address = email_link.text
                # Imprimer l'adresse e-mail
                #email_link.click()
                data_list.append({'Adresse e-mail': email_address, 'Nom_assocaition': nom_association})
                # Fermez le navigat/eur
                driver.back()
                # Fermer la nouvelle fenêtre
                driver.close()
                
                # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
                driver.switch_to.window(driver.window_handles[0])
                
                # Basculer vers la fenêtre principale
                k+=1
           
        except NoSuchElementException:
            driver.back()
            # Fermer la nouvelle fenêtre
            driver.close()
            
            # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
            driver.switch_to.window(driver.window_handles[0])
            
            # Basculer vers la fenêtre principale
            k+=1
            
        except TimeoutException:
            driver.back()
            # Fermer la nouvelle fenêtre
            driver.close()
            
            # Une fois que vous avez terminé avec la nouvelle fenêtre, vous pouvez basculer vers la fenêtre principale
            driver.switch_to.window(driver.window_handles[0])
            
            # Basculer vers la fenêtre principale
            k+=1
            
    # Créer un DataFrame à partir de data_list
    df =  pandas.DataFrame(data_list)

    # Charger le DataFrame existant
    df_base_de_donnees =  pandas.read_excel('..\Associations\Recherche_clients_associations\Base_de_données.xlsx')

    # Supprimer les adresses en commun de df 
    df = df[~df['Adresse e-mail'].isin(df_base_de_donnees['Adresse e-mail'])]


    # Charger le DataFrame existant
    df_base_de_donnees =  pandas.read_excel('..\Associations\Recherche_clients_associations\Mails_disponibles.xlsx'
    )

    # Supprimer les adresses en commun de df Prospect_potentiels_Volcanos
    df = df[~df['Adresse e-mail'].isin(df_base_de_donnees['Adresse e-mail'])]


    # Supprimer les doublons basés sur la colonne 'Adresse e-mail'
    df_sans_doublons = df.drop_duplicates(subset=['Adresse e-mail'], keep='first')


    # Enregistrer le DataFrame mis à jour dans un nouveau fichier Excel
    df_sans_doublons.to_excel('..\Associations\Recherche_clients_associations\Resultats_code.xlsx', index=False)
    
    print('quoi')
    passer = driver.find_element(By.XPATH, '//*[@id="results"]/div[1]/button[4]')
    driver.implicitly_wait(10)
    time.sleep(0.2)
    driver.execute_script("arguments[0].click();", passer)
    i+=1
print('quoii')
# Créer un DataFrame à partir de data_list
df = pandas.DataFrame(data_list)

# Charger le DataFrame existant pour la base de données
df_base_de_donnees = pandas.read_excel('..\Associations\Recherche_clients_associations\Base_de_données.xlsx')

# Supprimer les adresses en commun de df
df = df[~df['Adresse e-mail'].isin(df_base_de_donnees['Adresse e-mail'])]

# Charger le DataFrame existant pour les mails disponibles
df_mails_disponibles = pandas.read_excel('..\Associations\Recherche_clients_associations\Mails_disponibles.xlsx')

# Supprimer les adresses en commun de df
df = df[~df['Adresse e-mail'].isin(df_mails_disponibles['Adresse e-mail'])]

# Supprimer les doublons basés sur la colonne 'Adresse e-mail'
df_sans_doublons = df.drop_duplicates(subset=['Adresse e-mail'], keep='first')

# Enregistrer le DataFrame mis à jour dans un nouveau fichier Excel
df_sans_doublons.to_excel('..\Associations\Recherche_clients_associations\Resultats_code.xlsx', index=False)




ModuleNotFoundError: No module named 'pandas'